<a href="https://colab.research.google.com/github/Andresflorezdev/DAT-05_ANALISIS_y_Visualizacion_de_Datos-Esumer-Medellin/blob/main/Laboratorio_Pandas_Estudiantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas: DataFrames y diagnóstico de calidad
## Laboratorio de estudiantes y matrículas
**ESUMER / Estud-IA · Módulo 2

Tiempo sugerido: 60 minutos. Datos totalmente sintéticos, sin información de personas reales.

**Objetivo:** auditar un archivo, proponer reglas, aplicar una limpieza básica y explicar sus efectos. No buscamos que todos los campos queden completos a cualquier costo.

### Uso en Google Colab
1. Abre este archivo `.ipynb` desde **Archivo > Subir notebook**.
2. Sube `Matriculas_diagnostico.csv` al panel **Archivos** de Colab. Conserva su nombre. La celda de carga también ofrece un selector si no encuentra el archivo.
3. Ejecuta las celdas en orden. Completa las marcadas con `# COMPLETA AQUÍ`.
4. Si reinicias el entorno, vuelve a subir los datos. Guarda una copia de tu notebook.

**Buenas prácticas:** conservar la fuente, trabajar en una copia, leer errores completos, revisar las filas afectadas y documentar cada decisión. No publiques datos personales de tus proyectos.


## 1. Contexto y reglas del caso
Una institución ficticia reúne matrículas de dos periodos. Quiere contar matrículas por programa y describir edades y valores. Antes debe saber qué registros puede usar.

**Unidad de observación:** una matrícula de un estudiante en un periodo. Un estudiante puede aparecer varias veces. `id_matricula` debe identificar una matrícula única. El índice de Pandas solo ubica filas.

| Campo | Significado y regla del ejercicio |
|---|---|
| id_matricula | Identificador textual único de matrícula. No convertirlo a número. |
| id_estudiante | Identificador textual. Se repite si hay varias matrículas. |
| periodo | 2026-1 o 2026-2. |
| programa | Analítica, Marketing, Desarrollo Web, Administración o Especialización. |
| sede | Robledo, Centro o Virtual. |
| edad | Años cumplidos. Rango permitido para este caso: 16 a 80. Es una regla ficticia, no universal. |
| fecha_matricula | Fecha ISO AAAA-MM-DD. Debe ser anterior o igual a fecha_inicio en este caso. |
| fecha_inicio | Inicio del periodo. Fecha ISO. |
| estado | Activa, Cancelada o Finalizada. |
| valor_matricula | Valor bruto en COP, positivo. Especialización puede costar 8.000.000 COP. |
| descuento_pct | Porcentaje entre 0 y 100, expresado en puntos porcentuales: 20 significa 20 %. |
| beca | Sí o No. Un faltante significa desconocido, no No. |

**Equivalencias confirmadas por la institución ficticia:** Mercadeo = Marketing; Administracion = Administración. Espacios y mayúsculas pueden normalizarse. “No registrado” requiere validación y no se asigna a otro programa.

Las claves repetidas con valores contradictorios se separan para validar. No hay evidencia para decidir cuál versión es correcta.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
ruta_csv = '/content/drive/MyDrive/Analisis de Datos Esumer/Clase 7/Matriculas_diagnostico.csv' #Agregar la ruta de drive
ruta_excel = '/content/drive/MyDrive/Analisis de Datos Esumer/Clase 7/Matriculas_diagnostico.xlsx' #Agregar la ruta de drive

In [3]:
from pathlib import Path
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, 'to_string') else obj)
print('Versión de Pandas:', pd.__version__)
archivo = Path(ruta_csv)
if not archivo.exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Coloca el CSV junto a este notebook antes de ejecutarlo.')
df_original = pd.read_csv(archivo, encoding='utf-8-sig',
                          dtype={'id_matricula':'string', 'id_estudiante':'string'})
df = df_original.copy(deep=True)
display(df.head())

Versión de Pandas: 2.2.3


,id_matricula,id_estudiante,periodo,programa,sede,edad,fecha_matricula,fecha_inicio,estado,valor_matricula,descuento_pct,beca
0,M0001,E001,2026-1,analítica,Robledo,58,2026-01-08,2026-01-25,Activa,1200000,20.0,Sí
1,M0002,E002,2026-1,MARKETING,Centro,33,2026-01-12,2026-01-25,Cancelada,1400000,0.0,No
2,M0003,E003,2026-1,Desarrollo Web,Virtual,NaN,2026-01-07,2026-01-25,Finalizada,1800000,0.0,No
3,M0004,E004,2026-1,Administración,Robledo,19,2026-01-07,2026-01-25,Activa,1400000,10.0,No
4,M0005,E005,2026-1,Analítica,Centro,50,2026-01-05,2026-01-25,Cancelada,$ 1.250.000,30.0,No


### Lectura alternativa de Excel
Si subes `Matriculas_diagnostico.xlsx`, puedes usar:
```python
df_excel = pd.read_excel('Matriculas_diagnostico.xlsx', sheet_name='Matriculas',
                        dtype={'id_matricula':'string','id_estudiante':'string'})
```
CSV es texto delimitado. Excel puede contener varias hojas. Ambos se convierten en DataFrames. Los tipos inferidos pueden diferir: siempre revísalos. En un CSV, `sep` debe coincidir con el separador del archivo.


In [4]:
df_excel = pd.read_excel(ruta_excel, sheet_name='Matriculas',
                        dtype={'id_matricula':'string','id_estudiante':'string'})

df_excel

,id_matricula,id_estudiante,periodo,programa,sede,edad,fecha_matricula,fecha_inicio,estado,valor_matricula,descuento_pct,beca
0,M0001,E001,2026-1,analítica,Robledo,58,2026-01-08,2026-01-25,Activa,1200000,20.0,Sí
1,M0002,E002,2026-1,MARKETING,Centro,33,2026-01-12,2026-01-25,Cancelada,1400000,0.0,No
2,M0003,E003,2026-1,Desarrollo Web,Virtual,NaN,2026-01-07,2026-01-25,Finalizada,1800000,0.0,No
3,M0004,E004,2026-1,Administración,Robledo,19,2026-01-07,2026-01-25,Activa,1400000,10.0,No
4,M0005,E005,2026-1,Analítica,Centro,50,2026-01-05,2026-01-25,Cancelada,$ 1.250.000,30.0,No
...,...,...,...,...,...,...,...,...,...,...,...,...
118,M0119,E059,2026-2,Desarrollo Web,Centro,51,2026-07-20,2026-07-25,Cancelada,1400000,20.0,No
119,M0120,E060,2026-2,Administración,Virtual,46,2026-07-07,2026-07-25,Finalizada,1600000,10.0,No
120,M0046,E046,2026-1,Marketing,Robledo,54,2026-01-20,2026-01-25,Activa,1400000,30.0,Sí
121,M0047,E047,2026-1,Desarrollo Web,Centro,44,2026-01-11,2026-01-25,Cancelada,1200000,0.0,No


### Ejercicio 1. Tamaño y estructura
Obtén tamaño, nombres de columnas y tipos.

**Pista:** `shape` y `columns` son atributos. `info()` es un método y lleva paréntesis.

**Resultado esperado:** 123 filas y 12 columnas. edad y valor_matricula se leen como texto por los valores mezclados. info() imprime un resumen y devuelve None.

In [10]:
# COMPLETA AQUÍ
# Consulta shape, columns, dtypes e info().

#cantidad_filas = df_excel.shape[0]
#cantidad_columnas = df_excel.shape[1]
#cantidad_filas, cantidad_columnas

print(df_excel.shape)
print(df_excel.columns)
print(df_excel.info())

(123, 12)
Index(['id_matricula', 'id_estudiante', 'periodo', 'programa', 'sede', 'edad',
       'fecha_matricula', 'fecha_inicio', 'estado', 'valor_matricula',
       'descuento_pct', 'beca'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_matricula     123 non-null    string 
 1   id_estudiante    123 non-null    string 
 2   periodo          123 non-null    object 
 3   programa         123 non-null    object 
 4   sede             123 non-null    object 
 5   edad             115 non-null    object 
 6   fecha_matricula  123 non-null    object 
 7   fecha_inicio     122 non-null    object 
 8   estado           123 non-null    object 
 9   valor_matricula  122 non-null    object 
 10  descuento_pct    122 non-null    float64
 11  beca             122 non-null    object 
dtypes: float64(1), object(9), strin

### Series y DataFrames
Una Series es una estructura de una dimensión. Un DataFrame reúne columnas bajo un mismo índice.
```python
df['edad']       # Series
df[['edad']]     # DataFrame con una columna
```
La tabla tiene dos dimensiones: primero pensamos en filas y luego en columnas. `loc` selecciona por etiquetas y condiciones. `iloc` selecciona por posiciones.


In [17]:
print(df_excel.columns.tolist())
print(df_original.columns.tolist())

['id_matricula', 'id_estudiante', 'periodo', 'programa', 'sede', 'edad', 'fecha_matricula', 'fecha_inicio', 'estado', 'valor_matricula', 'descuento_pct', 'beca']
['id_matricula', 'id_estudiante', 'periodo', 'programa', 'sede', 'edad', 'fecha_matricula', 'fecha_inicio', 'estado', 'valor_matricula', 'descuento_pct', 'beca']


In [11]:
display(df[['id_matricula','id_estudiante','programa']].head(3))
display(df.iloc[0:3, 0:4])
display(df.loc[df['id_estudiante'].eq('E001'), ['id_matricula','periodo']])

,id_matricula,id_estudiante,programa
0,M0001,E001,analítica
1,M0002,E002,MARKETING
2,M0003,E003,Desarrollo Web


,id_matricula,id_estudiante,periodo,programa
0,M0001,E001,2026-1,analítica
1,M0002,E002,2026-1,MARKETING
2,M0003,E003,2026-1,Desarrollo Web


,id_matricula,periodo
0,M0001,2026-1
60,M0061,2026-2


### Ejercicio 2. Selección y filtros
Muestra matrícula, estudiante y programa de las filas cuyo periodo es 2026-2.

**Pista:** Una comparación genera una Series de True/False. loc conserva las filas True.

**Resultado esperado:** 60 filas del segundo periodo. El filtro no cambia df.

In [20]:
# COMPLETA AQUÍ
# segundo_periodo = df.loc[condicion, columnas]
segundo_periodo = df_excel.loc[
    df_excel['periodo'] == "2026-2",
    ['id_matricula', 'id_estudiante', 'programa']
]

print(segundo_periodo)
print("\nFilas esperadas: ",len(segundo_periodo))

    id_matricula id_estudiante        programa
60         M0061          E001       Analítica
61         M0062          E002       Marketing
62         M0063          E003  Desarrollo Web
63         M0064          E004  Administración
64         M0065          E005       Analítica
65         M0066          E006       Marketing
66         M0067          E007  Desarrollo Web
67         M0068          E008  Administración
68         M0069          E009       Analítica
69         M0070          E010       Marketing
70         M0071          E011  Desarrollo Web
71         M0072          E012  Administración
72         M0073          E013       Analítica
73         M0074          E014       Marketing
74         M0075          E015  Desarrollo Web
75         M0076          E016  Administración
76         M0077          E017       Analítica
77         M0078          E018       Marketing
78         M0079          E019  Desarrollo Web
79         M0080          E020  Administración
80         M0

In [23]:
display(df.describe(include='all'))

,id_matricula,id_estudiante,periodo,programa,sede,edad,fecha_matricula,fecha_inicio,estado,valor_matricula,descuento_pct,beca
count,123,123,123,123,123,115,123,122,123,122,122.000000,122
unique,120,60,2,11,5,44,33,2,4,9,NaN,2
top,M0051,E051,2026-1,Desarrollo Web,Virtual,22,2026-01-11,2026-01-25,Cancelada,1400000,NaN,No
freq,2,3,63,31,41,7,7,62,41,36,NaN,96
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.286885,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.997234,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.000000,NaN


`describe()` resume variables numéricas por defecto. `include='all'` incluye otras columnas. `count` cuenta valores no nulos. Las columnas de texto muestran frecuencias, aunque representen números. Estos resúmenes no detectan todas las contradicciones.

## 2. Auditoría antes de modificar
Dimensiones: completitud (faltantes), unicidad (llaves), validez (formato y rango) y consistencia (categorías y relaciones entre campos).


### Ejercicio 3. Nulos visibles
Calcula cantidad y porcentaje de nulos por columna.

**Pista:** Cada True cuenta como 1. La media de una máscara calcula la proporción.

**Resultado esperado:** edad: 8 nulos; valor_matricula: 1; descuento_pct: 1; fecha_inicio: 1; beca: 1. Los espacios y s/d aún no cuentan como nulos.

In [30]:
# COMPLETA AQUÍ
# Usa isna().sum() e isna().mean() * 100.

display(df_excel.isna().sum())

display(df_excel.isna().mean()*100)

,0
id_matricula,0
id_estudiante,0
periodo,0
programa,0
sede,0
edad,8
fecha_matricula,0
fecha_inicio,1
estado,0
valor_matricula,1


,0
id_matricula,0.000000
id_estudiante,0.000000
periodo,0.000000
programa,0.000000
sede,0.000000
edad,6.504065
fecha_matricula,0.000000
fecha_inicio,0.813008
estado,0.000000
valor_matricula,0.813008


In [31]:
edad_texto = df['edad'].astype('string')
display(df.loc[edad_texto.str.strip().isin(['','s/d']), ['id_matricula','sede','edad']])
print('Nulos de edad en Virtual:', df.loc[df['sede'].eq('Virtual'), 'edad'].isna().sum())

,id_matricula,sede,edad
26,M0027,Virtual,s/d
29,M0030,Virtual,


Nulos de edad en Virtual: 8


**Reflexión:** los ocho nulos de edad están en Virtual. ¿Eliminar todas esas filas dejaría la misma representación de las sedes? ¿Qué significa reemplazar una edad por cero?
Respuesta orientativa: borrarlas reduce de manera desigual la presencia de Virtual. Cero inventaría una edad. Conservaremos el registro y dejaremos la edad como desconocida.


### Ejercicio 4. Duplicados con contexto
Cuenta repeticiones exactas y muestra todas las filas con id_matricula repetido.

**Pista:** `keep=False` marca todas las filas de una llave repetida. El conteo de copias adicionales usa keep="first" por defecto.

**Resultado esperado:** 2 copias exactas adicionales. 6 filas con llave repetida: M0046, M0047 y M0051. M0051 tiene valores contradictorios.

In [32]:
# COMPLETA AQUÍ
# Compara duplicated() con duplicated(subset=['id_matricula'], keep=False).

print(df_excel.duplicated().sum())
print(df_excel[df_excel.duplicated(subset=['id_matricula'], keep=False)])

2
    id_matricula id_estudiante periodo        programa     sede edad  \
45         M0046          E046  2026-1       Marketing  Robledo   54   
46         M0047          E047  2026-1  Desarrollo Web   Centro   44   
50         M0051          E051  2026-1  Desarrollo Web  Virtual   30   
120        M0046          E046  2026-1       Marketing  Robledo   54   
121        M0047          E047  2026-1  Desarrollo Web   Centro   44   
122        M0051          E051  2026-1  Desarrollo Web  Virtual   30   

    fecha_matricula fecha_inicio      estado valor_matricula  descuento_pct  \
45       2026-01-20   2026-01-25      Activa         1400000           30.0   
46       2026-01-11   2026-01-25   Cancelada         1200000            0.0   
50       2026-01-11   2026-01-25  Finalizada         1800000           10.0   
120      2026-01-20   2026-01-25      Activa         1400000           30.0   
121      2026-01-11   2026-01-25   Cancelada         1200000            0.0   
122      2026-01-11

**Pregunta:** ¿podemos borrar repeticiones de `id_estudiante`? No. El mismo estudiante puede tener otra matrícula. Un duplicado depende de la unidad de observación y la llave.


In [33]:
display(df['programa'].value_counts(dropna=False))
display(df['sede'].value_counts(dropna=False))

,count
programa,
Desarrollo Web,31
Marketing,29
Administración,29
Analítica,27
analítica,1
MARKETING,1
Desarrollo web,1
Mercadeo,1
Administracion,1


,count
sede,
Virtual,41
Robledo,40
Centro,40
robledo,1
CENTRO,1


### Ejercicio 5. Conversión exploratoria de edad
Convierte sin cambiar el original e identifica entradas que no se pudieron convertir.

**Pista:** coerce convierte lo no interpretable a NaN. Eso debe auditarse, no ocultarse.

**Resultado esperado:** 2 fallos nuevos (s/d y un espacio). 2 edades fuera de rango (9 y 150).

In [36]:
nulos_antes = df_excel['edad'].isna().sum()
edad_prueba = pd.to_numeric(df_excel['edad'], errors='coerce')

print('Nulos Antes: ', nulos_antes)
print('Nulos despues: ', edad_prueba.isna().sum())

Nulos Antes:  8
Nulos despues:  10


In [35]:
# COMPLETA AQUÍ
# edad_prueba = pd.to_numeric(..., errors='coerce')
# Compara los nulos antes y después.

display(df_excel.loc[df_excel['edad'].notna() & edad_prueba.isna(), ['id_matricula', 'sede', 'edad']])
display(df_excel.loc[(edad_prueba < 9) | (edad_prueba > 150), ['id_matricula','sede','edad']])

,id_matricula,sede,edad
26,M0027,Virtual,s/d
29,M0030,Virtual,


,id_matricula,sede,edad


## 3. Decisiones de tratamiento
Completa una breve bitácora antes de limpiar. Puedes proponer otra decisión si explicas su evidencia e impacto.

| Problema | Tratamiento de referencia | Qué se conserva |
|---|---|---|
| Copias exactas | Conservar una versión | Copias descartadas en archivo separado |
| Llave contradictoria | Separar ambas versiones para validación | Registros completos pendientes |
| Categorías equivalentes | Normalizar con mapa confirmado | Fuente original |
| Edad inválida o faltante | Mantener matrícula, edad desconocida | Valor fuente e indicador de revisión |
| Dinero con símbolo y separadores | Convertir según formato colombiano documentado | Valor fuente |
| Valor negativo / descuento inválido | Marcar y dejar el campo desconocido | Valor fuente e indicador |
| Fechas inválidas o contradictorias | Marcar, no inventar fecha | Registro y valores fuente |
| Valor alto autorizado | Conservar | Contexto de Especialización |

**Tu decisión:** ¿imputarías algún campo? ¿Con qué evidencia? No usaremos imputación automática en la solución base.


In [38]:
# Eliminación controlada de copias exactas. La confirmación del caso permite hacerlo.
copias_exactas = df.loc[df.duplicated()].copy()
trabajo = df.drop_duplicates().copy()
conflicto_llave = trabajo.duplicated('id_matricula', keep=False)
pendientes_llave = trabajo.loc[conflicto_llave].copy()
limpio = trabajo.loc[~conflicto_llave].copy()
print('Fuente:',len(df), 'Copias:',len(copias_exactas),
      'Pendientes por llave:',len(pendientes_llave), 'Base de trabajo:',len(limpio))

Fuente: 123 Copias: 2 Pendientes por llave: 2 Base de trabajo: 119


### Ejercicio 6. Normalización confirmada
Normaliza programa con strip/lower y el mapa. Guarda primero el valor fuente.

**Pista:** map devuelve nulo para valores sin equivalencia. No los asignamos al programa más frecuente.

**Resultado esperado:** 5 categorías válidas y 1 programa pendiente, procedente de No registrado.

In [39]:
# Conserva programa_fuente y crea el mapa de equivalencias definido en el caso.
limpio['programa_fuente'] = limpio['programa']

# Estandarizamos el texto (sin espacios adicionales al inicio/final y en minúsculas)
programa_limpio = limpio['programa'].astype('string').str.strip().str.lower()

# Definimos el diccionario de equivalencias en minúsculas para facilitar el mapeo
mapa_equivalencias = {
    'analítica': 'Analítica',
    'analitica': 'Analítica',
    'marketing': 'Marketing',
    'mercadeo': 'Marketing',
    'desarrollo web': 'Desarrollo Web',
    'administración': 'Administración',
    'administracion': 'Administración',
    'especialización': 'Especialización',
    'especializacion': 'Especialización'
}

# Aplicamos el mapa de equivalencias
limpio['programa'] = programa_limpio.map(mapa_equivalencias)

# Creamos una bandera para identificar aquellos registros con programas no reconocidos (ej. 'No registrado')
limpio['revisar_programa'] = limpio['programa'].isna()

# Visualizamos el resultado de las categorías resultantes
print(limpio['programa'].value_counts(dropna=False))

programa
Marketing          30
Administración     30
Desarrollo Web     29
Analítica          28
NaN                 1
Especialización     1
Name: count, dtype: int64


In [43]:
for col in ['sede','estado']:
    limpio[col+'_fuente'] = limpio[col]
    limpio[col] = limpio[col].astype('string').str.strip().str.title()

### Ejercicio 7. Edad y validación de rango
Conserva la edad original, convierte, marca faltantes o valores fuera de rango y deja esos campos como desconocidos.

**Pista:** La máscara reúne faltantes y valores fuera del rango del caso.

**Resultado esperado:** 12 edades requieren revisión. Se conservan las matrículas y edad_fuente.

In [45]:
# COMPLETA AQUÍ
# Usa to_numeric, between y loc. No elimines las matrículas.

limpio['edad_fuente'] = limpio['edad']

# Convertimos a numérico usando pd.to_numeric
edad_num = pd.to_numeric(limpio['edad'], errors='coerce')

# Según las reglas del caso, el rango permitido de edad es de 16 a 80 años.
# Los nulos o valores fuera de este rango se consideran inválidos.
mascara = edad_num.isna() | (edad_num < 16) | (edad_num > 80)

# Creamos la bandera para el consolidado de incidencias
limpio['revisar_edad'] = mascara

# Dejamos como desconocido (NaN) en la columna limpia aquellos con problemas, sin eliminar el registro
limpio['edad'] = edad_num.mask(mascara)

display(limpio.loc[mascara, ['id_matricula', 'edad_fuente', 'edad']])

,id_matricula,edad_fuente,edad
2,M0003,NaN,NaN
5,M0006,NaN,NaN
8,M0009,NaN,NaN
11,M0012,NaN,NaN
14,M0015,NaN,NaN
17,M0018,NaN,NaN
20,M0021,NaN,NaN
23,M0024,NaN,NaN
26,M0027,NaN,NaN
29,M0030,NaN,NaN


### Valores monetarios
El caso usa COP enteros. El texto `$ 1.250.000` significa 1250000. Los puntos son separadores de miles en esta fuente, no decimales. No apliques esta regla a otras fuentes sin verificar su formato.


In [46]:
limpio['valor_matricula_fuente'] = limpio['valor_matricula']
valor_texto = limpio['valor_matricula'].astype('string')
valor_texto = valor_texto.str.replace('$','',regex=False).str.replace(' ','',regex=False).str.replace('.','',regex=False)
valor_convertido = pd.to_numeric(valor_texto, errors='coerce')
limpio['revisar_valor'] = valor_convertido.isna() | valor_convertido.le(0)
limpio['valor_matricula'] = valor_convertido.mask(limpio['revisar_valor'])
display(limpio.loc[limpio['revisar_valor'], ['id_matricula','valor_matricula_fuente']])
display(limpio.loc[limpio['programa'].eq('Especialización'), ['programa','valor_matricula']])

,id_matricula,valor_matricula_fuente
7,M0008,pendiente
10,M0011,-50000
13,M0014,NaN


,programa,valor_matricula
16,Especialización,8000000


### Ejercicio 8. Descuento
Marca porcentajes inválidos o faltantes. Conserva el original y no supongas que un descuento desconocido equivale a cero.

**Pista:** 0 es un descuento conocido del 0 %. Un faltante tiene otro significado.

**Resultado esperado:** 3 descuentos pendientes: 120, -5 y un faltante.

In [47]:
# COMPLETA AQUÍ
# Valida descuento_pct entre 0 y 100 y conserva descuento_pct_fuente.

# 1. Conservamos el valor original para mantener la trazabilidad
limpio['descuento_pct_fuente'] = limpio['descuento_pct']

# 2. Convertimos la columna a valores numéricos de forma segura
descuento_num = pd.to_numeric(limpio['descuento_pct'], errors='coerce')

# 3. Creamos una máscara para identificar valores fuera de rango (0 a 100) o nulos
mascara_descuento = descuento_num.isna() | (descuento_num < 0) | (descuento_num > 100)

# 4. Creamos la bandera lógica para la revisión de incidencias
limpio['revisar_descuento'] = mascara_descuento

# 5. Guardamos en la columna limpia los valores numéricos válidos (los inválidos quedan como NaN)
limpio['descuento_pct'] = descuento_num.mask(mascara_descuento)

# Visualizamos los registros que presentaron algún problema en el descuento
display(limpio.loc[mascara_descuento, ['id_matricula', 'descuento_pct_fuente', 'descuento_pct']])

,id_matricula,descuento_pct_fuente,descuento_pct
21,M0022,120.0,NaN
24,M0025,-5.0,NaN
27,M0028,NaN,NaN


In [48]:
for col in ['fecha_matricula','fecha_inicio']:
    limpio[col+'_fuente'] = limpio[col]
    limpio[col] = pd.to_datetime(limpio[col], format='%Y-%m-%d', errors='coerce')
limpio['revisar_fecha'] = (limpio['fecha_matricula'].isna() | limpio['fecha_inicio'].isna()
                         | limpio['fecha_matricula'].gt(limpio['fecha_inicio']))
display(limpio.loc[limpio['revisar_fecha'], ['id_matricula','fecha_matricula_fuente','fecha_inicio_fuente']])
# Se conserva la fecha contradictoria y se excluye de análisis de plazos hasta validarla.
limpio['revisar_beca'] = limpio['beca'].isna()


,id_matricula,fecha_matricula_fuente,fecha_inicio_fuente
30,M0031,2026-02-30,2026-01-25
33,M0034,2026-02-05,2026-01-25
36,M0037,2026-01-09,NaN


## 4. Antes y después
Las incidencias pueden solaparse: una fila puede tener más de un problema. No sumes incidencias para contar personas ni registros afectados.
La base preparada conserva campos pendientes. Cada análisis debe definir cuáles necesita válidos.


In [49]:
banderas = ['revisar_programa','revisar_edad','revisar_valor','revisar_descuento','revisar_fecha','revisar_beca']
display(limpio[banderas].sum().rename('Incidencias pendientes'))
print('Filas con al menos una incidencia:',limpio[banderas].any(axis=1).sum())
resumen = pd.DataFrame({
 'medida':['Filas','Llaves repetidas (filas)','Nulos visibles de edad','Categorías de programa (sin nulos)'],
 'antes':[len(df),df.duplicated('id_matricula',keep=False).sum(),df['edad'].isna().sum(),df['programa'].nunique()],
 'después':[len(limpio),limpio.duplicated('id_matricula',keep=False).sum(),limpio['edad'].isna().sum(),limpio['programa'].nunique()]})
display(resumen)
print('Conciliación:',len(df), '=',len(copias_exactas),'+',len(pendientes_llave),'+',len(limpio))
assert len(df)==len(copias_exactas)+len(pendientes_llave)+len(limpio)
assert not limpio['id_matricula'].duplicated().any()
pd.testing.assert_frame_equal(df_original,df)


,Incidencias pendientes
revisar_programa,1
revisar_edad,12
revisar_valor,3
revisar_descuento,3
revisar_fecha,3
revisar_beca,1


Filas con al menos una incidencia: 23


,medida,antes,después
0,Filas,123,119
1,Llaves repetidas (filas),6,0
2,Nulos visibles de edad,8,12
3,Categorías de programa (sin nulos),11,5


Conciliación: 123 = 2 + 2 + 119


### Ejercicio 9. Efecto sobre un indicador
Compara la media de edades conocidas válidas con la media que obtendrías rellenando los faltantes con cero.

**Pista:** mean omite nulos. El resultado describe únicamente los registros con edad conocida.

**Resultado esperado:** 107 edades conocidas de 119 registros. Rellenar con cero reduce la media y cambia su significado.

In [50]:
# COMPLETA AQUÍ
# Compara mean() con fillna(0).mean(). Solo simula, no cambies la columna.

# 1. Calculamos la media de edad ignorando los valores desconocidos
media_real = limpio['edad'].mean()
cantidad_validos = limpio['edad'].notna().sum()

# 2. Calculamos la media de edad rellenando temporalmente los nulos con 0
media_con_ceros = limpio['edad'].fillna(0).mean()

# Imprimimos los resultados para analizar el sesgo
print(f"Cantidad de estudiantes con edad conocida: {cantidad_validos}")
print(f"Media de edad (ignorando nulos): {media_real:.2f} años")
print(f"Media de edad (rellenando con 0): {media_con_ceros:.2f} años")
print(f"Diferencia / Sesgo introducido: {media_real - media_con_ceros:.2f} años")

Cantidad de estudiantes con edad conocida: 107
Media de edad (ignorando nulos): 37.21 años
Media de edad (rellenando con 0): 33.45 años
Diferencia / Sesgo introducido: 3.75 años


### Conexión con groupby (opcional, fuera del reto principal)
`value_counts()` cuenta matrículas por categoría. `groupby()` separa filas por una categoría y después aplica una operación a cada grupo. No cambia por sí solo la fuente.
```python
limpio.groupby('programa', dropna=False)['edad'].mean()
```
Esto calcula la media de edades conocidas por programa, no una media de todas las matrículas. Lo desarrollaremos en la clase de agrupaciones.

## 5. Entrega y conclusiones
En este taller conserva tu notebook y responde:
1. ¿Qué representa una fila y cuál es su llave?

  R/: la fila y llave representan una matricula de un estudiante; su llave es `id_matricula`

2. ¿Cuáles fueron los problemas de mayor impacto?

  R/: duplicados con datos contradictorios, edades con texto (s/d) y montos de dinero no numericos

3. ¿Qué regla aplicaste y qué evidencia la respalda?

  R/: Validar la edad exclusivamente entre los 16 y los 80 años segun el lineamiento institucional

4. ¿Qué campos quedaron pendientes y qué análisis limitan?

  R/: Edad, programa y valores invalidos; limita crear estadisticas demograficas y de ingresos exactos

5. ¿Qué sesgo produciría borrar filas o rellenar campos automáticamente?

R/: Borrar filas subrepresenta la sede virtual; rellenar con ceros baja artificialmente la edad promedio de 37 a 33 años

Un aumento de nulos puede reflejar que identificamos valores inválidos antes ocultos. El criterio de éxito es la trazabilidad y la validez, no tener cero nulos.


In [51]:
# Exportaciones opcionales para conservar la trazabilidad en Colab.
limpio.to_csv('Matriculas_preparadas.csv', index=False, encoding='utf-8-sig')
pendientes_llave.to_csv('Matriculas_pendientes_llave.csv', index=False, encoding='utf-8-sig')
copias_exactas.to_csv('Matriculas_copias_exactas.csv', index=False, encoding='utf-8-sig')
print('Exportados: base preparada, conflictos de llave y copias exactas.')

Exportados: base preparada, conflictos de llave y copias exactas.


## Proyecto final y fuentes de datos
Individual o grupos de máximo 4. Exposición de máximo 10 minutos el 27 de noviembre, Masterclass presencial. Entrega: presentación con contexto, proceso, hallazgos y limitaciones. No se requiere entregar datos ni aplicar todo el curso. Protege información personal y confidencial.

Los mejores trabajos podrán presentarse en la semana de la innovación. El programa también incluye un bootcamp posterior, cuyo contenido no se define aquí.

Fuentes para explorar: [Datos Abiertos Colombia](https://www.datos.gov.co/), [DANE](https://www.dane.gov.co/index.php/estadisticas-por-tema), [Banco Mundial](https://data.worldbank.org/indicator), [Kaggle](https://www.kaggle.com/datasets) y [UCI](https://archive.ics.uci.edu/datasets).
Antes de elegir: pregunta viable, variables suficientes, diccionario, periodo y cobertura, licencia y tamaño manejable.

Referencias técnicas: [Pandas: datos faltantes](https://pandas.pydata.org/docs/user_guide/missing_data.html), [to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html), [duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html), [indexación](https://pandas.pydata.org/docs/user_guide/indexing.html).
